# Phase 6 — Streamlit Frontend

**Goal:** A clean, no-frills UI where you can upload a PDF, ask questions, see the answer with citations, and inspect the retrieved chunks.

Streamlit is the right choice here because:

- One Python file, no JavaScript, no build step.
- File upload, chat input, and Markdown rendering are one-liners.
- Sufficiently polished for portfolio/demo use; not pretending to be a SaaS frontend.

By the end you will have a working `apps/streamlit/app.py` that talks to the FastAPI backend from Phase 5.


## 6.1 — UI design

```
+--------------------------------------------------------+
|  Multimodal AI Research & Teaching Assistant           |
+--------------------------------------------------------+
| [ Upload PDF ]   docs: 1 (15 pages, 32 chunks)         |
+--------------------------------------------------------+
| Mode: ( Default | Beginner | Graduate | Interview |    |
|         Quiz me | Explain figure )                     |
+--------------------------------------------------------+
|  Q: What is multi-head attention?            [ Ask ]   |
+--------------------------------------------------------+
|  Answer                                                |
|  ----                                                  |
|  Multi-head attention runs scaled dot-product          |
|  attention in parallel over h projections [page 4]...  |
|                                                        |
|  Citations: page 4, page 5                             |
|                                                        |
|  ▸ Show retrieved chunks                               |
+--------------------------------------------------------+
```

Two columns on wider screens; collapse to one column on mobile.


## 6.2 — The full `streamlit_app.py`

Below is the file. Save it to `apps/streamlit/app.py` and run with:

```bash
streamlit run apps/streamlit/app.py
```


In [ ]:
streamlit_src = r'''
"""apps/streamlit/app.py — single-file UI for the RAG assistant."""
import httpx
import streamlit as st

API = "http://localhost:8000"

st.set_page_config(page_title="Research & Teaching Assistant", layout="wide")
st.title("Multimodal AI Research & Teaching Assistant")

# --- sidebar: upload + doc list ---------------------------------------------
with st.sidebar:
    st.header("Documents")
    uploaded = st.file_uploader("Upload a PDF", type=["pdf"])
    if uploaded is not None and st.button("Index this PDF"):
        with st.spinner("Parsing and indexing..."):
            r = httpx.post(f"{API}/upload",
                           files={"file": (uploaded.name, uploaded.getvalue(), "application/pdf")},
                           timeout=600)
        if r.status_code == 200:
            st.success(f"Indexed {r.json()['n_pages']} pages, {r.json()['n_chunks']} chunks")
        else:
            st.error(f"Upload failed: {r.text}")

    st.divider()
    try:
        docs = httpx.get(f"{API}/documents", timeout=5).json()
    except Exception:
        docs = []
        st.warning("Backend not reachable. Start it with: uvicorn apps.api.main:app --reload")
    for d in docs:
        st.write(f"- {d['source']} ({d['n_pages']}p, {d['n_chunks']}c)")

# --- main panel: ask --------------------------------------------------------
mode = st.radio(
    "Mode",
    ["Default", "Beginner", "Graduate", "Interview", "Quiz me", "Explain figure"],
    horizontal=True,
)
mode_prefix = {
    "Default":     "",
    "Beginner":    "Explain like I am new to this topic. ",
    "Graduate":    "Explain at the level of a graduate student in ML. ",
    "Interview":   "Explain as you would in an ML system-design interview. ",
    "Quiz me":     "Generate 5 multiple-choice quiz questions (with answers) about: ",
    "Explain figure": "Explain the figure(s) on the relevant page(s) and what they show. ",
}[mode]

question = st.text_input("Ask a question about the indexed documents", "")
k = st.slider("Top-k retrieved chunks", 1, 10, 5)

if st.button("Ask", type="primary", disabled=not question):
    payload = {"question": mode_prefix + question, "k": k}
    with st.spinner("Thinking..."):
        try:
            r = httpx.post(f"{API}/ask", json=payload, timeout=120)
            r.raise_for_status()
            resp = r.json()
        except Exception as e:
            st.error(f"Error: {e}")
            st.stop()

    st.subheader("Answer")
    st.markdown(resp["answer"])

    if resp["cited_pages"]:
        st.caption("Cited pages: " + ", ".join(str(p) for p in resp["cited_pages"]))
    st.caption(f"Model: {resp['model']}  |  Latency: {resp['latency_s']:.1f}s")

    with st.expander("Retrieved chunks"):
        for r_ in resp["retrieved"]:
            st.markdown(f"**page {r_['page']}** — score {r_['score']:.3f}  \\n_{r_['chunk_id']}_")
            st.markdown(f"> {r_['preview']}")
            st.divider()
'''
print(streamlit_src[:1200])


## 6.3 — Running the full stack

Two terminals:

```bash
# Terminal A
uvicorn apps.api.main:app --reload --port 8000

# Terminal B
streamlit run apps/streamlit/app.py
```

Streamlit picks an open port (usually 8501) and opens your browser automatically.


## 6.4 — A few UX details worth getting right

1. **Show retrieval scores.** Users learn to spot bad retrieval when scores are all under 0.3.
2. **Surface the cited pages.** That is the credibility hook of the whole app — make it visible, not buried.
3. **Disable "Ask" while a request is in flight** (`disabled=` + spinner). Streamlit reruns top-to-bottom on every interaction; without this, users will spam the button.
4. **Persist conversation history with `st.session_state`** if you want multi-turn. We keep it single-turn for the MVP because grounded one-shot answers are the differentiator.
5. **Show a page preview** when a citation is clicked. We will add this in the figure notebook (Phase 7).


## 6.5 — Embedding screenshots in the README

For the README's "demo" section, capture:

1. A wide screenshot showing the upload + ask flow.
2. A close-up of an answer with cited pages.
3. The retrieved-chunks expander open.

Save under `docs/screenshots/` and reference them with relative paths in `README.md`.


## What you learned

- A complete single-file Streamlit UI that talks to the FastAPI backend.
- A "teaching modes" radio that pre-pends a system-style prefix to the user's question (we will formalize these prompts in Phase 8).
- How to keep the UX honest by showing retrieval scores and citation pages.

## Exercises

1. Add multi-turn chat using `st.chat_message` and `st.session_state["history"]`.
2. Add a "Download answer + citations as Markdown" button.
3. When a cited page number is shown, render a thumbnail of that page next to the answer using PyMuPDF's `page.get_pixmap`.

## Next: [Phase 7 — Figure extraction & VLM](./2026-05-25-phase07-figure-extraction-and-vlm.ipynb)
